In [1]:
# load libaries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as sm

The tool outputs are created by the script "merge_tool_outputs.ipynb".\
The tools have max capacity and therefore the data set with antibodies had to be split up for predictions to be performed. To reduce risk of intefering with the merging of the files, to make each script shorter and easier to understand and to be able to remove the originals files this computation of scores part and the merge tool outputs part have been separated. 

In [6]:
# Load merged tool outputs
#netMHC1_pep9 =
#netMHC1_pep14 =
#netMHC_II_pep15 =
#netMHC_II_pep12 =
waltz = pd.read_csv("merged_tool_outputs/waltz_merged_tool_outputs.csv")
biophi_relaxed = pd.read_csv("merged_tool_outputs/biophi_relaxed_merged_tool_outputs.csv")
biophi_strict = pd.read_csv("merged_tool_outputs/biophi_strict_merged_tool_outputs.csv")

# loda file with ADA scores for each antibody
ADA = pd.read_csv("ADA_218AB.csv")

# Compute scores
netMHCpan tools give one score for each peptide - HLA allele combination
Therefore, for each antibody a immunogenicity score will be calculated. The definition of what score is considerd immunogenetic differs for the different tools \
for each netMHC score of interest: precentile score, immunogenicity socre and pre processing score

In [ ]:
# netMHC1_EL_pep9 percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep9_percentile = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_percentile')
    )

# netMHC1_EL_pep9 Immunogenicity score 

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep9_immunogenicity_score = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_immunogenicity_score')
    )

# netMHC1_EL_pep9 Preprocessing score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMCH1_pep9_preProcess = netMHC1_EL_pep9.groupby('sequence name')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep9_preProcess'})

# netMHC1_EL_pep14

# Percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep14_percentile = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_percentile')
    )

# Immunogenicity score

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep14_immunogenicity_score = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_immunogenicity_score')
    )

# Pre-proocessing score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 

netMCH1_pep14_preProcess = netMHC1_EL_pep14.groupby('sequence name')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep14_preProcess'})


# netMHC_II_EL_pep12

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep12_percentile = (
    netMHC_II_EL_pep12.assign(immunogenic=netMHC_II_EL_pep12['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep12_percentile')
    )

# Immunogenicity score

# remove the rows with the immunogenicity score of '-' before calculating the mean
netMHC_II_EL_pep12 = netMHC_II_EL_pep12[netMHC_II_EL_pep12['immunogenicity score'] != '-']
# make the column with the immunogenicity score into a numeric column
netMHC_II_EL_pep12['immunogenicity score'] = pd.to_numeric(netMHC_II_EL_pep12['immunogenicity score'])

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody.
netMHC_II_pep12_immunogenicity_score = netMHC_II_EL_pep12.groupby('sequence name')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep12_immunogenicity_score'})


# Does not have pre processing score. Due to tool not being able to predict pre processing on peptides shorter than 13 amino acids


# netMHC_II_EL_pep15

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep15_percentile = (
    netMHC_II_EL_pep15.assign(immunogenic=netMHC_II_EL_pep15['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('sequence name')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep15_percentile')
    )

# Immunogenicity score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_immunogenicity_score = netMHC_II_EL_pep15.groupby('sequence name')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep15_immunogenicity_score'})

# Pre-proocessing score
# MHC class 2 has 2 preprocessing scores of interest: mhcii-np cleavage probability score and mhcii-np cleavage probability percentile rank

# mhcii-np cleavage probability

# remove the rows with the cleavage probability score of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability score'] != '-']
# make the column with the cleavage probability score into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability score'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability score'])
# Compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProb = netMHC_II_EL_pep15.groupby('sequence name')['mhcii-np cleavage probability score'].mean().reset_index().rename(columns={'processing total score': 'netMHC_II_pep15_preProcess_cleavProb'})

# mhcii-np cleavage probability percentile rank

# remove the rows with the cleavage probability percentile rank of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] != '-']
# make the column with the cleavage probability percentile rank into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'])
# compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProbPercentile = netMHC_II_EL_pep15.groupby('sequence name')['mhcii-np cleavage probability percentile rank'].mean().reset_index().rename(columns={'mhcii-np cleavage probability percentile rank': 'netMHC_II_pep15_preProcess_cleavProbPercentile'})

# rename the column '...' to nr_aggs
waltz = waltz.rename(columns={'...': 'nr_aggs'})

# compute the total number of amio acids that are considerd immunogenetic
def sum_ranges(s):
    if pd.isna(s):
        return 0
    total = 0
    for part in s.split(';'):
        start, end = part.strip().split('-')
        total += int(end) - int(start) + 1 # beacuse the values are inclusive
    return total

waltz['nr_aggs'] = waltz['waltz_score'].apply(sum_ranges)

In [ ]:
# Merge scores from all predictors into one df

# create df with all the dfs that shall me merged into one
dfs_to_merge = [
    netMHC1_pep9_percentile,
    netMHC1_pep9_immunogenicity_score,
    netMCH1_pep9_preProcess,
    netMHC1_pep14_percentile,
    netMHC1_pep14_immunogenicity_score,
    netMCH1_pep14_preProcess,
    netMHC_II_pep12_percentile,
    netMHC_II_pep12_immunogenicity_score,
    netMHC_II_pep15_percentile,
    netMHC_II_pep15_immunogenicity_score,
    netMHC_II_pep15_preProcess_cleavProb,
    netMHC_II_pep15_preProcess_cleavProbPercentile,
]

# Rename the sequence name column to antibody to make the merging easier. 
dfs_to_merge = [df.rename(columns={'sequence name': 'antibody'}) for df in dfs_to_merge]

# Create the df with all scores, start with the ADA scores
all_predictors_andADA = ADA

# Merge all dfs on antibody name
for df in dfs_to_merge:
    all_predictors_andADA = all_predictors_andADA.merge(df, on='antibody', how='left')

# remove all whitespaces in antobdy column
all_predictors_andADA['antibody'] = all_predictors_andADA['antibody'].str.strip()

# Then merge the three last dfs
all_predictors_andADA = all_predictors_andADA.merge(waltz[['antibody','nr_aggs']], on='antibody', how='left').rename(columns={'nr_aggs': 'waltz_nr_aggs'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_KabKabRelaxed, on='antibody', how='left')
all_predictors_andADA = all_predictors_andADA.merge(biophi_KabKabStrict, on='antibody', how='left')

all_predictors_andADA

In [ ]:
# Make all white space and '-' in the column names to '_'
all_predictors_andADA.columns = (
    all_predictors_andADA.columns
    .str.replace(r'[\s\-]+', '_', regex=True)
)

In [ ]:
# export score table
all_predictors_andADA.to_csv("all_predictors_217AB(biophidata).csv")